# Batch Review — Historical Video Reprocessing

Reprocess Pi-captured videos through YOLOv11, auto-flag low-confidence and
anomalous detections, and upload flagged frames to Roboflow for human review.

**Requirements:** Google Colab with GPU runtime (T4 recommended).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics easyocr roboflow


In [ ]:
import os, re
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────
MODEL_PATH = '/content/drive/MyDrive/Fun Project/Cat monitor/model/fair_feeder_v13_yolov11s.pt'
VIDEO_DIR  = '/content/drive/MyDrive/Fun Project/Cat monitor/TAPO_autoupload'
OUTPUT_DIR = '/content/drive/MyDrive/Fun Project/Cat monitor/Test_postmodel_output'

# ── Video selection ────────────────────────────────────────────────
# Run 1 (recommended first): FEEDING_WINDOW_ONLY = True
#   Processes only 06:18-06:30 clips (~15-20 videos).
#   All 5 classes present (kibble, cats, bowl). Highest retraining value.
#
# Run 2: FEEDING_WINDOW_ONLY = False
#   Processes a random sample of non-feeding clips for Dan/Sanbo diversity.
#   MAX_VIDEOS caps the run so Colab doesn't time out.
# Dates to skip entirely (YYYYMMDD) — e.g. days you already manually labeled in Roboflow
EXCLUDE_DATES        = []      # e.g. ['20260326', '20260327']
FEEDING_WINDOW_ONLY  = True   # True = 06:18-06:30 clips only; False = all clips
FEEDING_WINDOW_START = '0618'  # HHMM
FEEDING_WINDOW_END   = '0630'  # HHMM (inclusive)
MAX_VIDEOS           = 50      # Only used when FEEDING_WINDOW_ONLY = False

# ── YOLO parameters ────────────────────────────────────────────────
CONF_THRESHOLD = 0.45
IOU_THRESHOLD  = 0.20
IMGSZ          = 1280
FRAME_SKIP     = 7

# ── Flagging thresholds ────────────────────────────────────────────
FLAG_CONF_THRESHOLD  = 0.40
FLAG_BLIP_MAX_FRAMES = 2
FLAG_BLIP_GAP_FRAMES = 5
FLAG_IOU_CONFLICT    = 0.50
FLAG_KIBBLE_JUMP     = 15
FLAG_DEDUP_WINDOW    = 3

# ── Roboflow credentials ───────────────────────────────────────────
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = ''

ROBOFLOW_WORKSPACE = 'test-7vyqo'

# Upload tracking — prevents re-uploading frames already sent to Roboflow
TRACKING_FILE = str(Path(OUTPUT_DIR) / 'roboflow_uploaded.txt')
ROBOFLOW_PROJECT   = 'ir-kibble'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded.')
print(f'  Mode: {"Feeding window only (06:18-06:30)" if FEEDING_WINDOW_ONLY else f"All clips (max {MAX_VIDEOS})"}')


In [ ]:
import random
from ultralytics import YOLO
from pathlib import Path

model = YOLO(MODEL_PATH)
print(f'Model loaded: {MODEL_PATH}')

VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv'}
video_dir = Path(VIDEO_DIR)
all_videos = sorted(
    p for p in video_dir.iterdir()
    if p.suffix.lower() in VIDEO_EXTS
)
print(f'Found {len(all_videos)} total video(s) in folder.')

def _in_feeding_window(filename, start='0618', end='0630'):
    """Return True if motion_YYYYMMDD_HHMMSS_... filename is within HHMM window."""
    m = re.match(r'motion_\d{8}_(\d{6})', filename)
    if not m:
        return False
    hhmm = m.group(1)[:4]  # first 4 chars of HHMMSS
    return start <= hhmm <= end

def _get_date(filename):
    """Extract YYYYMMDD from motion_YYYYMMDD_HHMMSS filename."""
    m = re.match(r'motion_(\d{8})_', filename)
    return m.group(1) if m else None

if FEEDING_WINDOW_ONLY:
    video_files = [v for v in all_videos
        if _in_feeding_window(v.name, FEEDING_WINDOW_START, FEEDING_WINDOW_END)
        and _get_date(v.name) not in EXCLUDE_DATES]
    print(f'Feeding window filter ({FEEDING_WINDOW_START}-{FEEDING_WINDOW_END}): {len(video_files)} clip(s)')
else:
    non_feeding = [v for v in all_videos
        if not _in_feeding_window(v.name, FEEDING_WINDOW_START, FEEDING_WINDOW_END)
        and _get_date(v.name) not in EXCLUDE_DATES]
    if len(non_feeding) > MAX_VIDEOS:
        random.seed(42)
        video_files = sorted(random.sample(non_feeding, MAX_VIDEOS))
        print(f'Non-feeding clips: sampled {MAX_VIDEOS} from {len(non_feeding)} (seed=42)')
    else:
        video_files = non_feeding
        print(f'Non-feeding clips: {len(video_files)} (all, under MAX_VIDEOS cap)')

print()
print('Videos to process:')
for vf in video_files:
    size_mb = vf.stat().st_size / (1024 * 1024)
    print(f'  {vf.name}  ({size_mb:.1f} MB)')
if EXCLUDE_DATES:
    print(f'Excluding dates: {EXCLUDE_DATES}')

In [ ]:
import cv2
import pickle
from tqdm.auto import tqdm

for vf in video_files:
    cache_path = Path(OUTPUT_DIR) / (vf.stem + '_detections.pkl')
    if cache_path.exists():
        print(f'Cache exists, skipping: {vf.name}')
        continue

    cap = cv2.VideoCapture(str(vf))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_cache = []
    frame_idx = 0

    print(f'Processing: {vf.name} ({total_frames} frames, skip={FRAME_SKIP})')
    pbar = tqdm(total=total_frames // FRAME_SKIP, desc=vf.stem)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % FRAME_SKIP != 0:
            frame_idx += 1
            continue

        results = model.predict(
            frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
            imgsz=IMGSZ, rect=True, verbose=False,
        )

        detections = []
        for r in results:
            for box in r.boxes:
                detections.append({
                    'class_id': int(box.cls[0]),
                    'class_name': model.names[int(box.cls[0])],
                    'conf': float(box.conf[0]),
                    'x1': int(box.xyxy[0][0].item()),
                    'y1': int(box.xyxy[0][1].item()),
                    'x2': int(box.xyxy[0][2].item()),
                    'y2': int(box.xyxy[0][3].item()),
                })

        _, jpeg_buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
        frames_cache.append({
            'frame_idx': frame_idx,
            'detections': detections,
            'jpeg': jpeg_buf.tobytes(),
        })

        pbar.update(1)
        frame_idx += 1

    cap.release()
    pbar.close()

    with open(cache_path, 'wb') as f:
        pickle.dump(frames_cache, f)
    print(f'  Cached {len(frames_cache)} frames -> {cache_path.name}')

print('Phase 1 complete.')


In [ ]:
import sys
import pickle
from pathlib import Path

sys.path.insert(0, '/content/drive/MyDrive/Fun Project/Cat monitor/fair-feeder')
from flagging import flag_detections

all_flagged = {}  # video_stem -> list of flagged frames

for vf in video_files:
    cache_path = Path(OUTPUT_DIR) / (vf.stem + '_detections.pkl')
    if not cache_path.exists():
        print(f'No cache for {vf.name}, skipping flagging.')
        continue

    with open(cache_path, 'rb') as f:
        frames = pickle.load(f)

    flagged = flag_detections(
        frames,
        conf_threshold=FLAG_CONF_THRESHOLD,
        blip_max_frames=FLAG_BLIP_MAX_FRAMES,
        blip_gap_frames=FLAG_BLIP_GAP_FRAMES,
        iou_conflict=FLAG_IOU_CONFLICT,
        kibble_jump=FLAG_KIBBLE_JUMP,
        dedup_window=FLAG_DEDUP_WINDOW,
    )
    all_flagged[vf.stem] = flagged

    tag_counts = {}
    for ff in flagged:
        for tag in ff.get('tags', []):
            tag_counts[tag] = tag_counts.get(tag, 0) + 1

    print(f'{vf.name}: {len(flagged)} flagged frames')
    for tag, count in sorted(tag_counts.items()):
        print(f'  {tag}: {count}')

print(f'\nTotal flagged across all videos: {sum(len(v) for v in all_flagged.values())}')


In [ ]:
from roboflow_upload import upload_flagged_frames

if not ROBOFLOW_API_KEY:
    print('No ROBOFLOW_API_KEY set. Skipping upload.')
    print('Set it in Colab Secrets or paste it in the Config cell.')
else:
    combined_uploaded = 0
    combined_failed = 0

    for stem, flagged in all_flagged.items():
        if not flagged:
            continue
        print(f'Uploading {len(flagged)} frames from {stem}...')
        result = upload_flagged_frames(
            flagged,
            api_key=ROBOFLOW_API_KEY,
            workspace=ROBOFLOW_WORKSPACE,
            project=ROBOFLOW_PROJECT,
            video_stem=stem,
            tracking_file=TRACKING_FILE,
        )
        combined_uploaded += result.uploaded
        combined_failed += result.failed
        skipped = getattr(result, 'skipped', 0)
        print(f'  Uploaded: {result.uploaded}, Skipped: {skipped} (already done), Failed: {result.failed}')

    print(f'\nTotal uploaded: {combined_uploaded}, Total failed: {combined_failed}')


In [ ]:
print('=' * 50)
print('BATCH REVIEW SUMMARY')
print('=' * 50)
print(f'Total videos processed: {len(video_files)}')
total_flagged = sum(len(v) for v in all_flagged.values())
print(f'Total flagged frames:   {total_flagged}')
print()

print('Per-video breakdown:')
for vf in video_files:
    flagged = all_flagged.get(vf.stem, [])
    tag_counts = {}
    for ff in flagged:
        for tag in ff.get('tags', []):
            tag_counts[tag] = tag_counts.get(tag, 0) + 1
    tags_str = ', '.join(f'{t}: {c}' for t, c in sorted(tag_counts.items())) or 'none'
    print(f'  {vf.name}: {len(flagged)} flagged  [{tags_str}]')

print()
print('Next steps:')
print('  1. Open Roboflow and review the uploaded flagged frames')
print('  2. Correct or confirm the auto-generated labels')
print('  3. Generate a new dataset version in Roboflow')
print('  4. Retrain the model with the expanded dataset')
